# Runtime Analysis
Reads directly from pkl files.

**Input:** 6 pkl files from 30-run experiments  
**Output:** `Runtime_Analysis.xlsx` (3 sheets)

## Cell 1: Imports

In [3]:
import pickle
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
print("Imports OK")

Imports OK


## Cell 2: Load pkl

In [5]:
# Load all pkl files
print("Loading pkl files...")
with open('Strategy1_30runs_PartA.pkl', 'rb') as f: s1a = pickle.load(f)
with open('Strategy1_30runs_PartB.pkl', 'rb') as f: s1b = pickle.load(f)
with open('S2A_results_kmf_30runs.pkl', 'rb') as f: s2a_kmf = pickle.load(f)
with open('S2A_results_ccf_30runs.pkl', 'rb') as f: s2a_ccf = pickle.load(f)
with open('S2B_results_kmf_30runs.pkl', 'rb') as f: s2b_kmf = pickle.load(f)
with open('S2B_results_ccf_30runs.pkl', 'rb') as f: s2b_ccf = pickle.load(f)
print("All loaded ✓")

Loading pkl files...
All loaded ✓


## Cell 3: Extract Runtime

In [7]:
# =============================================================================
# Extract runtime_sec from pkl files
# S1 structure: pkl['clustering_results_30']['approach2'][ds][method]['runtime_sec']
# S2 structure: pkl['final_results_30'][ds][method]['runtime_sec']
# =============================================================================

METHOD_LABEL = {
    'kmeans':              'K-Means',
    'fairlet':             'Fairlet',
    'bfkm':               'BFKM',
    'fair_centroid':       'CCF',
    'postprocessing_nfp':  'PP-NFP',
    'postprocessing_gini': 'PP-Gini',
    'rawlsian':            'Rawlsian',
}
METHODS = ['K-Means','Fairlet','BFKM','CCF','PP-NFP','PP-Gini','Rawlsian']

DS_LABEL = {
    'adult':'D1', 'compas':'D2', 'german':'D3', 'credit':'D4', 'law':'D5',
    'adult_race':'D1_race', 'adult_combined':'D1_comb',
    'compas_race':'D2_race', 'compas_combined':'D2_comb',
    'law_race':'D5_race',   'law_combined':'D5_comb',
    'diabetes_gender':'D6', 'diabetes_race':'D6_race', 'diabetes_combined':'D6_comb',
    'dutch_gender':'D7',
    'meps_gender':'D8',     'meps_race':'D8_race',     'meps_combined':'D8_comb',
}
DS_ORDER = list(DS_LABEL.keys())
SLOW     = {'CCF', 'Rawlsian'}   # 15 runs instead of 30

def extract_runtime_s1(pkl):
    """S1: clustering_results_30 may have 'approach2' key or be flat."""
    results = pkl['clustering_results_30']
    first   = list(results.keys())[0]
    if first in ['approach1','approach2']:
        results = results.get('approach2', results.get('approach1', {}))
    rt = {}
    for ds, methods in results.items():
        rt[ds] = {}
        for m, r in methods.items():
            label = METHOD_LABEL.get(m, m)
            rt[ds][label] = r.get('runtime_sec', np.nan)
    return rt

def extract_runtime_s2(pkl):
    """S2: final_results_30[ds][method]['runtime_sec']"""
    results = pkl['final_results_30']
    rt = {}
    for ds, methods in results.items():
        rt[ds] = {}
        for m, r in methods.items():
            label = METHOD_LABEL.get(m.lower(), m)
            rt[ds][label] = r.get('runtime_sec', np.nan)
    return rt

# Merge S1
rt_s1 = {}
for pkl in [s1a, s1b]:
    for ds, mv in extract_runtime_s1(pkl).items():
        rt_s1[ds] = mv

# Merge S2 (combine kmf + ccf for each part)
rt_s2 = {}
for pkl in [s2a_kmf, s2a_ccf, s2b_kmf, s2b_ccf]:
    for ds, mv in extract_runtime_s2(pkl).items():
        if ds not in rt_s2: rt_s2[ds] = {}
        rt_s2[ds].update(mv)

print(f"S1 datasets: {len(rt_s1)}  S2 datasets: {len(rt_s2)}")
print(f"\nSample S1 adult runtime (seconds, total across all runs):")
for m, s in rt_s1.get('adult', {}).items():
    n = 15 if m in SLOW else 30
    print(f"  {m:15s}: total={s:7.1f}s  per_run={s/n:.1f}s  (n={n})")

S1 datasets: 18  S2 datasets: 18

Sample S1 adult runtime (seconds, total across all runs):
  K-Means        : total=  399.7s  per_run=13.3s  (n=30)
  Fairlet        : total= 1928.8s  per_run=64.3s  (n=30)
  BFKM           : total=  405.4s  per_run=13.5s  (n=30)
  CCF            : total= 8743.2s  per_run=582.9s  (n=15)
  PP-NFP         : total=  361.1s  per_run=12.0s  (n=30)
  PP-Gini        : total= 4299.1s  per_run=143.3s  (n=30)
  Rawlsian       : total=  335.1s  per_run=22.3s  (n=15)


## Cell 4: Compute Statistics

In [9]:
# =============================================================================
# Compute per-run runtime and summary statistics
# =============================================================================

def per_run(method, total_s):
    """Divide by number of runs (15 for slow methods, 30 for others)."""
    n = 15 if method in SLOW else 30
    return total_s / n if not np.isnan(total_s) else np.nan

# Build per-run DataFrames
def build_runtime_df(rt_dict, label):
    rows = []
    for ds in DS_ORDER:
        if ds not in rt_dict: continue
        mv   = rt_dict[ds]
        row  = {'Dataset': DS_LABEL.get(ds, ds)}
        for m in METHODS:
            total = mv.get(m, np.nan)
            row[m] = round(per_run(m, total), 1) if not np.isnan(total) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

df_s1 = build_runtime_df(rt_s1, 'S1')
df_s2 = build_runtime_df(rt_s2, 'S2')

# Average per method
print("=== Average per-run runtime (seconds) ===")
print(f"{'Method':15s}  {'S1 avg':>8s}  {'S2 avg':>8s}  {'vs K-Means (S1)':>16s}")
km_s1 = df_s1['K-Means'].mean()
km_s2 = df_s2['K-Means'].mean()
for m in METHODS:
    s1_avg = df_s1[m].mean()
    s2_avg = df_s2[m].mean() if m in df_s2.columns else np.nan
    ratio  = s1_avg / km_s1
    print(f"  {m:15s}  {s1_avg:8.1f}  {s2_avg:8.1f}  {ratio:>14.1f}×")

print(f"\n(K-Means baseline: S1={km_s1:.1f}s/run, S2={km_s2:.1f}s/run)")

=== Average per-run runtime (seconds) ===
Method             S1 avg    S2 avg   vs K-Means (S1)
  K-Means              38.1       7.2             1.0×
  Fairlet             141.3      23.8             3.7×
  BFKM                 50.3       6.7             1.3×
  CCF                1139.1     212.6            29.9×
  PP-NFP               34.8       6.8             0.9×
  PP-Gini             186.4      71.6             4.9×
  Rawlsian             70.5      12.8             1.8×

(K-Means baseline: S1=38.1s/run, S2=7.2s/run)


## Cell 5: Write Excel

In [13]:
# =============================================================================
# Write Excel: Runtime Analysis Table
# =============================================================================

thin   = Side(style='thin', color='CCCCCC')
bdr    = Border(left=thin, right=thin, top=thin, bottom=thin)
H_FILL = PatternFill('solid', start_color='2E74B5')
A_FILL = PatternFill('solid', start_color='F5F9FD')
R_FILL = PatternFill('solid', start_color='FFE7E7')   # red = slowest
G_FILL = PatternFill('solid', start_color='E7FFE7')   # green = fastest
NO_FILL= PatternFill(fill_type=None)

def write_runtime_sheet(ws, df, title):
    ws.title = title
    headers = ['Dataset'] + METHODS + ['Fastest','Slowest','Ratio']

    # Header
    for ci, h in enumerate(headers, 1):
        c = ws.cell(1, ci, h)
        c.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
        c.fill      = H_FILL
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border    = bdr
    ws.row_dimensions[1].height = 28

    for ri, row in enumerate(df.itertuples(index=False), 2):
        alt = (ri % 2 == 0)
        bg  = A_FILL if alt else NO_FILL
        row_vals = {}
        for m in METHODS:
            try:
                row_vals[m] = df.loc[df['Dataset']==row.Dataset, m].iloc[0]
            except:
                row_vals[m] = np.nan
        num_vals = {m: v for m, v in row_vals.items() if not np.isnan(v)}
        fastest  = min(num_vals, key=num_vals.get) if num_vals else None
        slowest  = max(num_vals, key=num_vals.get) if num_vals else None
        ratio    = (num_vals[slowest]/num_vals[fastest]) if fastest and slowest else None

        # Dataset col
        c = ws.cell(ri, 1, row.Dataset)
        c.font = Font(name='Arial', size=9, bold=True)
        c.fill = bg; c.border = bdr
        c.alignment = Alignment(horizontal='center', vertical='center')

        # Method cols
        for ci, m in enumerate(METHODS, 2):
            v       = row_vals.get(m, np.nan)
            is_fast = (m == fastest)
            is_slow = (m == slowest)
            c = ws.cell(ri, ci, round(v,1) if not np.isnan(v) else '—')
            c.font      = Font(name='Arial', size=9, bold=(is_slow or is_fast))
            c.fill      = G_FILL if is_fast else (R_FILL if is_slow else bg)
            c.alignment = Alignment(horizontal='center', vertical='center')
            c.border    = bdr

        # Fastest / Slowest / Ratio
        for ci, val in enumerate([fastest or '—', slowest or '—',
                                   f"{ratio:.1f}×" if ratio else '—'],
                                  len(METHODS)+2):
            c = ws.cell(ri, ci, val)
            c.font = Font(name='Arial', size=9)
            c.fill = bg; c.border = bdr
            c.alignment = Alignment(horizontal='center', vertical='center')

        ws.row_dimensions[ri].height = 15

    # Average row
    avg_row = len(df) + 2
    c = ws.cell(avg_row, 1, 'Average')
    c.font = Font(name='Arial', bold=True, size=9)
    c.fill = PatternFill('solid', start_color='D6E4F0')
    c.border = bdr; c.alignment = Alignment(horizontal='center', vertical='center')
    for ci, m in enumerate(METHODS, 2):
        avg = df[m].mean()
        c = ws.cell(avg_row, ci, round(avg,1) if not np.isnan(avg) else '—')
        c.font = Font(name='Arial', bold=True, size=9)
        c.fill = PatternFill('solid', start_color='D6E4F0')
        c.border = bdr; c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[avg_row].height = 18

    # Column widths
    ws.column_dimensions['A'].width = 12
    for ci in range(2, len(headers)+2):
        ws.column_dimensions[get_column_letter(ci)].width = 11
    ws.freeze_panes = 'B2'

wb = Workbook()
wb.remove(wb.active)
write_runtime_sheet(wb.create_sheet('S1 Runtime'), df_s1, 'S1 Runtime')
write_runtime_sheet(wb.create_sheet('S2 Runtime'), df_s2, 'S2 Runtime')

# Combined average sheet
df_avg = df_s1.copy()
df_avg.iloc[:,1:] = (df_s1.iloc[:,1:].values + df_s2.iloc[:,1:].values) / 2
write_runtime_sheet(wb.create_sheet('Average S1+S2'), df_avg, 'Average S1+S2')

wb.save('Runtime_Analysis.xlsx')
print("✓ Saved: Runtime_Analysis.xlsx")
print("  Sheet 1: S1 Runtime (per-run seconds)")
print("  Sheet 2: S2 Runtime (per-run seconds)")
print("  Sheet 3: Average S1+S2")
print("  Green = fastest per row, Red = slowest per row")

C:\Users\G1237\AppData\Local\Temp\ipykernel_48112\3171451972.py:38: RuntimeWarning: divide by zero encountered in scalar divide
  ratio    = (num_vals[slowest]/num_vals[fastest]) if fastest and slowest else None


✓ Saved: Runtime_Analysis.xlsx
  Sheet 1: S1 Runtime (per-run seconds)
  Sheet 2: S2 Runtime (per-run seconds)
  Sheet 3: Average S1+S2
  Green = fastest per row, Red = slowest per row


In [15]:
# 在notebook里运行这个检查
with open('S2B_results_kmf_30runs.pkl', 'rb') as f:
    s2b = pickle.load(f)

print("S2B top keys:", list(s2b.keys()))
results = s2b['final_results_30']
first_ds = list(results.keys())[0]
first_m  = list(results[first_ds].keys())[0]
r = results[first_ds][first_m]
print(f"First ds: {first_ds}, method: {first_m}")
print("Result keys:", list(r.keys()))
print("runtime_sec:", r.get('runtime_sec', 'NOT FOUND'))

S2B top keys: ['majority_k_results', 'final_results_30', 'datasets', 'seeds', 'slow_seeds']
First ds: diabetes_gender, method: kmeans
Result keys: ['k', 'n_runs', 'mean_quality', 'std_quality', 'mean_fairness', 'std_fairness', 'all_quality', 'all_fairness']
runtime_sec: NOT FOUND
